# Week 16 – Research Assistant Agent with Tool Calls 16.3 Graded Mini Project
### Submitted by: Harley Dangan
### Date: 26 November 2026
### Submitted to: The Asian Institute of Management
### License: GNU General Public License v3.0 (GPL-3.0-or-later) - see LICENSE



## Load Environment Variables

This Jupyter notebook assumes that you have the environment variables configured. See .venv.example for reference.

**IMPORTANT NOTES:**

**langchain_ollama is used in this notebook. This notebook will only work with Ollama, Llama.cpp, or any similar inference engine.**

**The following command needs to be run in order to install the jailbreak guardrail models:**
**uv run python -c "from guardrails_ai.detect_jailbreak import DetectJailbreak; DetectJailbreak(on_fail='noop')"**

**Laya, a System One model, is also used in this project. It also needs to be configured in the environment variables. Host Laya locally or use the online API service.**

In [1]:
from dotenv import load_dotenv

# Read .env from the notebook's directory. Variables already set in the
# environment take precedence over the file.
load_dotenv()

import os

base_url = os.environ.get("BASE_URL")
llm= os.environ.get("LLM")
api_key = os.environ.get("OPENAI_API_KEY") # Actually, any string will work here since I used Ollama!

print("LLM configuration:")
if not base_url or not llm or not api_key:
    raise ValueError("Missing required environment variables: BASE_URL, LLM, API_KEY, or TAVILY_API_KEY")
print(f"Base URL: {base_url}")
print(f"LLM: {llm}")
print(f"API Key: {"set" if api_key else "not set"}")

print("\nLaya configuration:")
laya_base_url = os.environ.get("LAYA_BASE_URL")
laya_api_key = os.environ.get("LAYA_API_KEY")
laya_llm = os.environ.get("LLM")
if not laya_base_url or not laya_api_key or not laya_llm:
    raise ValueError("Missing required environment variables: LAYA_BASE_URL, LAYA_API_KEY, or LLM")
print(f"Laya Base URL: {laya_base_url}")
print(f"Laya LLM: {laya_llm}")
print(f"Laya API Key: {"set" if laya_api_key else "not set"}")

tavily_api_key = os.environ.get("TAVILY_API_KEY")
if not tavily_api_key:
    raise ValueError("Missing required environment variable: TAVILY_API_KEY")

print("\nLangsmith configuration:")
langsmith_api_key = os.environ.get("LANGSMITH_API_KEY")
if not langsmith_api_key:
    raise ValueError("Missing required environment variable: LANGSMITH_API_KEY")
print(f"LangSmith API Key: {"set" if langsmith_api_key else "not set"}")

LLM configuration:
Base URL: localhost:11434/v1
LLM: llama-3.2-1b-q4_k_m
API Key: set

Laya configuration:
Laya Base URL: localhost:11435/v1/systemone
Laya LLM: llama-3.2-1b-q4_k_m
Laya API Key: set

Langsmith configuration:
LangSmith API Key: set


## Initialize LLM

NOTE: Assumes local model running in Ollama.

In [2]:
from langchain_ollama import ChatOllama

# Initialize the ChatOllama model with the specified parameters
llm = ChatOllama(
    model=llm,
    temperature=0.3,
    base_url=base_url,
)

# Defining the system prompt for the model
system_prompt = """You are an expert Research Assistant Agent. Your core objective is to deliver comprehensive, highly accurate, and fact-verified research on any given topic. You have access to tools for searching the web and retrieving full webpage content.

### OPERATIONAL MANDATES
1. FACTUAL GROUNDING: Rely exclusively on verifiable information retrieved from your tools. If a fact cannot be sourced from your tool outputs, explicitly state that it is unavailable. Never assume, extrapolate, or hallucinate details.
2. LATENT SEARCH QUERIES: When an initial tool result is ambiguous, outdated, or incomplete, proactively formulate distinct follow-up queries or browse primary source URLs to fill knowledge gaps.
3. CONTRADICTION RESOLUTION: If different sources present conflicting numbers, dates, or accounts, outline both perspectives objectively. Do not pick a side unless one source is demonstrably more credible.
4. SYNTHESIS OVER SUMMARIZATION: Do not simply list summaries of search results. Synthesize the findings into a cohesive narrative structured with clear headers, logical groupings, and thematic continuity.

### TOOL-CALLING GUIDELINES
- Query Design: Keep search tool queries targeted and optimized. Avoid passing long conversational sentences to search engines. Use precise keywords, dates, or entity names.
- Deep Dive: If a search snippet provides a promising lead but lacks critical context, use the browser tool to read the full page before finalizing your synthesis.
- Temporal Context: Always keep the current year and context in mind to evaluate whether information is current or obsolete.

### OUTPUT STRUCTURE & STYLE
- Formality: Maintain a neutral, professional, and academic tone. Avoid conversational filler, personal pronouns, or fluff.
- Scannability: Use descriptive Markdown headers, bold key entities/metrics, and utilize bullet points for non-sequential data points.
- Citation: Always attach functional anchors or clear indicators mapping back to the specific source domain or URL retrieved by your tools."""



## Initialize Tools

In [3]:
import json
import os
import re
from urllib.parse import urlparse

from tavily import TavilyClient
import arxiv

# Instantiate the Tavily client once so the LangGraph nodes can call it.
tavily_client = TavilyClient(api_key=tavily_api_key)

# ---------------------------------------------------------------------------
# Tavily mock switch, driven by .env
#   TAVILY_MOCK=1 (default) -> nodes return canned results, no API credits used
#   TAVILY_MOCK=0           -> nodes call the real Tavily API
# ---------------------------------------------------------------------------
_FALSEY = {"0", "false", "no", "off", ""}
USE_MOCK_TAVILY = os.getenv("TAVILY_MOCK", "1").strip().lower() not in _FALSEY

print(f"Tavily search mode: {'MOCK (no API calls)' if USE_MOCK_TAVILY else 'LIVE (Tavily API)'}")

# Canned payload shaped like TavilyClient.search(). The `query` field is
# overwritten at call time so outputs still show what was actually asked.
MOCK_TAVILY_RESPONSE = {
    "mock": True,
    "answer": (
        "Mock summary: recent retrieval-augmented generation work focuses on better "
        "chunking, hybrid retrieval with reranking, and measuring how well answers "
        "stay grounded in the retrieved sources."
    ),
    "response_time": 0.42,
    "results": [
        {
            "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
            "url": "https://example.com/mock/rag-knowledge-intensive-nlp",
            "content": (
                "Mock document. Pairs a dense retriever with a seq2seq generator and reports "
                "gains on open-domain question answering and fact verification when the "
                "retriever is trained jointly with the generator."
            ),
            "score": 0.94,
        },
        {
            "title": "Hybrid Retrieval and Reranking for Grounded Generation",
            "url": "https://example.com/mock/hybrid-retrieval-reranking",
            "content": (
                "Mock document. Combines BM25 and dense vectors, then applies a cross-encoder "
                "reranker; the reranking stage accounts for most of the accuracy improvement "
                "on long-form answers."
            ),
            "score": 0.89,
        },
        {
            "title": "Evaluating Faithfulness in Retrieval-Augmented Systems",
            "url": "https://example.com/mock/evaluating-faithfulness",
            "content": (
                "Mock document. Introduces a benchmark that separates retrieval recall from "
                "citation accuracy, finding that models often cite plausible but unretrieved "
                "passages."
            ),
            "score": 0.83,
        },
        {
            "title": "Chunking Strategies for Scientific Literature Retrieval",
            "url": "https://example.com/mock/chunking-scientific-literature",
            "content": (
                "Mock document. Compares fixed-window, recursive and semantic chunking on arXiv "
                "corpora; semantic chunking improves answer completeness but increases index "
                "size substantially."
            ),
            "score": 0.77,
        },
    ],
}


def mock_tavily_search(query: str, max_results: int = 5) -> dict:
    """Return canned results in the same shape as TavilyClient.search()."""
    payload = dict(MOCK_TAVILY_RESPONSE)
    payload["query"] = query
    payload["results"] = MOCK_TAVILY_RESPONSE["results"][:max_results]
    return payload


def run_tavily_search(query: str, max_results: int = 5) -> dict:
    """Single dispatch point: mock data or the real API, depending on the switch."""
    if USE_MOCK_TAVILY:
        return mock_tavily_search(query=query, max_results=max_results)
    return tavily_client.search(query=query, max_results=max_results)


# Arxiv Search Function
# One reusable arXiv client. In arxiv>=4 the client (not the Search object)
# exposes .results(), so searches must go through arxiv_client.
arxiv_client = arxiv.Client()


def arxiv_search(query: str, max_results: int = 5) -> str:
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )
    
    results = []
    for result in arxiv_client.results(search):
        paper_info = (
            f"Title: {result.title}\n"
            f"Authors: {', '.join(author.name for author in result.authors)}\n"
            f"Published: {result.published.strftime('%Y-%m-%d')}\n"
            f"URL: {result.pdf_url}\n"
            f"Abstract: {result.summary}\n"
            f"{'-'*40}"
        )
        results.append(paper_info)
        
    return "\n".join(results) if results else "No papers found for this query."

# APA Citation Function
# Converts the raw output of tavily_search_node or arxiv_search_node into
# APA 7th edition references. The input format is detected automatically, and
# the source can also be forced with source="tavily" or source="arxiv".

_ARXIV_URL_RE = re.compile(r"arxiv\.org/(?:pdf|abs)/(.+?)(?:v\d+)?/?$")
_YEAR_RE = re.compile(r"\b(1[5-9]\d{2}|20\d{2})\b")


def _apa_author(full_name: str) -> str:
    """'Jane Q. Doe' -> 'Doe, J. Q.'"""
    parts = [part for part in re.split(r"\s+", full_name.strip()) if part]
    if not parts:
        return ""
    if len(parts) == 1:
        return parts[0]
    family = parts[-1]
    initials = " ".join(f"{part[0].upper()}." for part in parts[:-1])
    return f"{family}, {initials}"


def _apa_author_list(authors: str) -> str:
    """'Jane Doe, John Smith' -> 'Doe, J., & Smith, J.'"""
    people = [name for name in (_apa_author(a) for a in authors.split(",")) if name]
    if not people:
        return ""
    if len(people) == 1:
        return people[0]
    if len(people) == 2:
        return f"{people[0]}, & {people[1]}"
    return ", ".join(people[:-1]) + f", & {people[-1]}"


def _apa_year(text: str) -> str:
    match = _YEAR_RE.search(text or "")
    return match.group(1) if match else "n.d."


def _apa_web_reference(item: dict) -> str:
    """Tavily result -> APA reference for a web page."""
    title = (item.get("title") or "").strip().rstrip(".")
    url = (item.get("url") or "").strip()
    site = urlparse(url).netloc.removeprefix("www.") or "Web"
    year = _apa_year(item.get("published_date") or item.get("published") or "")
    # With no author, the page title takes the author position.
    title_part = f"{title}. " if title else ""
    return f"{title_part}({year}). {site}. {url}"


def _apa_preprint_reference(block: str) -> str:
    """One arXiv entry -> APA reference for a preprint."""
    head = block.split("Abstract:", 1)[0]
    fields = {}
    for line in head.splitlines():
        key, separator, value = line.partition(":")
        if separator:
            fields[key.strip().lower()] = value.strip()

    title = fields.get("title", "").strip().rstrip(".")
    authors = _apa_author_list(fields.get("authors", ""))
    year = _apa_year(fields.get("published", ""))
    url = fields.get("url", "")

    match = _ARXIV_URL_RE.search(url)
    link = f"https://doi.org/10.48550/arXiv.{match.group(1)}" if match else url

    if authors and not authors.endswith("."):
        authors += "."
    author_part = f"{authors} " if authors else ""
    return f"{author_part}({year}). {title}. arXiv. {link}"


def apa_convert(payload, source: str = "auto") -> str:
    """
    Convert search output from tavily_search_node or arxiv_search_node into
    APA style citations, one reference per paragraph.

    payload: raw ToolMessage content (a JSON string for Tavily, formatted text
             for arXiv), or the already-parsed dict/list.
    source:  "auto" (default), "tavily", or "arxiv".
    """
    if isinstance(payload, (dict, list)):
        data = payload
        text = json.dumps(payload)
    else:
        data = None
        text = str(payload)

    if isinstance(payload, str) and "No papers found" in payload:
        return "No citations to convert: the arXiv search returned no papers."

    if source == "auto":
        stripped = text.lstrip()
        if stripped.startswith(("{", "[")):
            source = "tavily"
        elif "Title:" in text and "Authors:" in text:
            source = "arxiv"
        else:
            raise ValueError(
                "Unrecognised search output: expected Tavily JSON or arXiv text."
            )

    if source == "tavily":
        if data is None:
            data = json.loads(text)
        results = data.get("results", []) if isinstance(data, dict) else data
        references = [
            _apa_web_reference(item)
            for item in results
            if item.get("url") or item.get("title")
        ]
    elif source == "arxiv":
        if "Title:" not in text or "Authors:" not in text:
            raise ValueError("Input does not look like arXiv search output.")
        blocks = [block.strip() for block in text.split("-" * 40) if block.strip()]
        references = [_apa_preprint_reference(block) for block in blocks]
    else:
        raise ValueError(f"Unknown source {source!r}; use 'auto', 'tavily' or 'arxiv'.")

    # dict.fromkeys removes duplicates while preserving the original order.
    references = [ref for ref in dict.fromkeys(references) if ref]
    if not references:
        return "No citations to convert."
    return "\n\n".join(references)


# 2. Map tool names to the actual functions
available_tools = {
    "arxiv_search": lambda query, max_results=5: arxiv_search(query=query, max_results=max_results),
    "tavily_search": lambda query, max_results=5: run_tavily_search(query=query, max_results=max_results),
    "apa_convert": lambda payload, source="auto": apa_convert(payload, source=source)
}

# Tool Schemas

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "tavily_search",
            "description": "Search the live web for current facts, news, and technical data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The specific search query optimized for a search engine. Do not include punctuation or conversational filler."
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "arxiv_search",
            "description": "Search for academic papers, preprints, and research articles on arXiv.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The scientific topic or paper keywords to search for.",
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "The number of research papers to return. Defaults to 3.",
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apa_convert",
            "description": "Convert raw Tavily or arXiv search results into APA style citations.",
            "parameters": {
                "type": "object",
                "properties": {
                    "payload": {
                        "type": "string",
                        "description": "Raw output from tavily_search or arxiv_search.",
                    },
                    "source": {
                        "type": "string",
                        "enum": ["auto", "tavily", "arxiv"],
                        "description": "Input format. Defaults to auto-detection.",
                    },
                },
                "required": ["payload"],
            },
        },
    }
]


Tavily search mode: MOCK (no API calls)


## Create LangGraph State Class

In [8]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

# State is one shared object every node reads from and writes to. 
# It is a TypedDict that defines the structure of the state object.
class AgentState(TypedDict):
    question: str
    answer: str
    threat_detected: bool
    research_route: str
    research_scores: dict
    research_reason: str
    messages: Annotated[list, add_messages]

## Define LangGraph Search Node Functions

In [9]:
from typing import Dict, List

from langchain_core.messages import ToolMessage

# The node signatures below use `State`, an alias for the AgentState TypedDict
# defined in the "Create LangGraph State Class" section.
State = AgentState

# Web Search Result node: This node takes the current state, which includes a question, and uses the LLM to generate an answer. It returns a dictionary containing the answer.
def tavily_search_node(state: State) -> Dict[str, List[ToolMessage]]:
    """
    LangGraph Node that executes the Tavily search tool call.
    """
    latest_message = state["messages"][-1]
    tool_messages = []
    
    if hasattr(latest_message, "tool_calls") and latest_message.tool_calls:
        for tool_call in latest_message.tool_calls:
            if tool_call["name"] == "tavily_search":
                try:
                    # Execute Tavily search (mock or live, see USE_MOCK_TAVILY)
                    query = tool_call["args"].get("query")
                    tool_output = run_tavily_search(query=query)
                    
                    # Convert response dictionary/list to a JSON string
                    if isinstance(tool_output, (dict, list)):
                        tool_output = json.dumps(tool_output)
                        
                except Exception as e:
                    tool_output = f"Error executing Tavily search: {str(e)}"
                
                tool_messages.append(
                    ToolMessage(
                        content=str(tool_output),
                        tool_call_id=tool_call["id"],
                        name=tool_call["name"]
                    )
                )
                
    return {"messages": tool_messages}

# Arxiv Search node: This node takes the current state, which includes a question, 
# and uses the LLM to generate an answer. It returns a dictionary containing the answer.
def arxiv_search_node(state: State) -> Dict[str, List[ToolMessage]]:
    """
    LangGraph Node that executes the arXiv search tool call.
    """
    latest_message = state["messages"][-1]
    tool_messages = []
    
    if hasattr(latest_message, "tool_calls") and latest_message.tool_calls:
        for tool_call in latest_message.tool_calls:
            if tool_call["name"] == "arxiv_search":
                try:
                    # Extract arguments passed by the LLM
                    args = tool_call["args"]
                    query = args.get("query")
                    max_results = args.get("max_results", 5) # Default to 5 if not provided
                    
                    # Execute the local arXiv search function
                    tool_output = arxiv_search(query=query, max_results=max_results)
                        
                except Exception as e:
                    tool_output = f"Error executing arXiv search: {str(e)}"
                
                tool_messages.append(
                    ToolMessage(
                        content=str(tool_output),
                        tool_call_id=tool_call["id"],
                        name=tool_call["name"]
                    )
                )
                
    return {"messages": tool_messages}

## Define LangGraph Start Edge

In [10]:
from uuid import uuid4

from langchain_core.messages import AIMessage

# Custom input for this run: edit the string, re-run this cell, then run the graph.
# Passing {"question": "..."} to app.invoke() overrides it for a single run.
CUSTOM_INPUT = "Recent advances in retrieval-augmented generation for scientific literature"


def start_node(state: AgentState) -> dict:
    """
    LangGraph START node.

    Turns the custom input into a single AIMessage holding one tool call per
    search tool, so the same question is handed to tavily_search_node and
    arxiv_search_node in parallel. Each node scans the latest message for the
    tool call carrying its own name.
    """
    question = (state.get("question") or CUSTOM_INPUT).strip()
    if not question:
        raise ValueError("CUSTOM_INPUT is empty - define the question for this run.")

    tool_calls = [
        {
            "name": "tavily_search",
            "args": {"query": question},
            "id": f"tavily-{uuid4().hex[:8]}",
            "type": "tool_call",
        },
        {
            "name": "arxiv_search",
            "args": {"query": question, "max_results": 5},
            "id": f"arxiv-{uuid4().hex[:8]}",
            "type": "tool_call",
        },
    ]

    return {
        "question": question,
        "messages": [AIMessage(content=question, tool_calls=tool_calls)],
    }


# Wire it up in the "Define LangGraph Edges" section:
# graph.add_node("start", start_node)
# graph.add_edge(START, "start")
# graph.add_edge("start", "tavily_search_node")
# graph.add_edge("start", "arxiv_search_node")


## Define LangGraph Security Guardrail Node

A conditional node placed between the start edge and the search nodes. It runs
`guardrails-ai-detect-prompt-injection` and `guardrails-ai-detect-jailbreak` on the
custom input. A flagged input is routed to a terminal node that reports the threat;
a clean input fans out to `tavily_search_node` and `arxiv_search_node`.

**NOTE: This block will load guardrail weights.**

In [7]:
import os

# Guardrails ships an OpenTelemetry exporter that retries an unreachable collector;
# disabling tracing keeps the notebook output clean.
os.environ.setdefault("OTEL_SDK_DISABLED", "true")

from guardrails import Guard
from guardrails.settings import settings as guardrails_settings

guardrails_settings.disable_tracing = True

from guardrails_ai.detect_jailbreak import DetectJailbreak
from guardrails_ai.detect_prompt_injection import DetectPromptInjection
from langgraph.types import Command

# Security layer: both validators run on the custom input before any tool is called.
# Guard().use() stores one validator list per target, so both go in a single call.
# DetectJailbreak downloads its HuggingFace models the first time this cell runs.
# check_vector=False keeps the injection validator on its local heuristics: its
# optional embedding check would call OpenAI, and this notebook runs on Ollama
# with a placeholder OPENAI_API_KEY.
security_guard = Guard().use(
    DetectPromptInjection(check_vector=False, on_fail="noop"),
    DetectJailbreak(on_fail="noop"),
)

THREAT_MESSAGE = (
    "Request blocked: the input was flagged as a prompt-injection or jailbreak attempt, "
    "so no searches were run."
)


def security_check_node(state: AgentState) -> Command:
    """
    Conditional security node.

    Clean input is passed to the research decision node, which picks one search
    tool. A hostile input is routed to the terminal threat_detected node instead,
    so no tool ever runs.
    """
    question = state.get("question", "")
    outcome = security_guard.validate(question)

    if not outcome.validation_passed:
        return Command(goto="threat_detected", update={"threat_detected": True})

    return Command(
        goto="research_decision_node",
        update={"threat_detected": False},
    )


def threat_detected_node(state: AgentState) -> dict:
    """Terminal node reached when the input is hostile: reports the block."""
    return {
        "answer": THREAT_MESSAGE,
        "messages": [AIMessage(content=THREAT_MESSAGE)],
    }


Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Define LangGraph Research Decision Node

Classifies the question with Laya and routes it to exactly one retrieval tool:
`tavily_search_node` for web search or `arxiv_search_node` for academic
literature. Laya returns a calibrated probability for each option and both are
printed with the margin, so the choice can be audited.


In [11]:
import os

import requests

# Laya (LAYA_* in .env) is a non-autoregressive decision model: instead of
# generating text it scores every option and returns calibrated probabilities,
# so the routing decision is a measured comparison rather than a guess.
laya_endpoint = os.getenv("LAYA_BASE_URL") or laya_base_url
if not laya_endpoint.startswith(("http://", "https://")):
    laya_endpoint = "http://" + laya_endpoint
laya_token = os.getenv("LAYA_API_KEY") or laya_api_key

# Routing rule: time-sensitive facts go to the web, research topics and papers
# go to arXiv. Laya caps each option at 48 tokens, so the criteria stay short.
ROUTE_OPTIONS = {
    "tavily_search_node": (
        "Time-sensitive information: recent events, announcements, current facts, news, prices, weather, sports, "
        "elections, releases, laws, regulations, current leaders and executives."
    ),
    "arxiv_search_node": (
        "Research papers and time-insensitive research topics: methods, theory, "
        "benchmarks, surveys, datasets, scientific explanations, paper definitions, prior search."
    ),
}

DECISION_QUESTION = {
    "route": {
        "type": "choice",
        "instructions": (
            "Which retrieval tool should answer? Use tavily for time-sensitive "
            "facts; use arxiv for research papers and time-insensitive research."
        ),
        "criteria": ROUTE_OPTIONS,
    }
}

_WEB_HINTS = (
    "news", "today", "latest", "now", "breaking", "price", "weather", "stock",
    "score", "election", "who is", "current", "trending", "release date",
)
_ACADEMIC_HINTS = (
    "paper", "arxiv", "study", "research", "benchmark", "survey", "theory",
    "dataset", "journal", "preprint", "literature", "state of the art",
)


def _ask_laya(question: str):
    """POST the routing question to Laya and return its calibrated probabilities."""
    headers = {"Content-Type": "application/json"}
    if laya_token:
        headers["Authorization"] = f"Bearer {laya_token}"

    response = requests.post(
        laya_endpoint,
        json={"state": {"question": question}, "questions": DECISION_QUESTION},
        headers=headers,
        timeout=30,
    )
    response.raise_for_status()
    probabilities = response.json()["answers"]["route"]["probabilities"]

    scores = {name: float(probabilities.get(name, 0.0)) for name in ROUTE_OPTIONS}
    total = sum(scores.values())
    if total <= 0:
        raise ValueError(f"Laya returned no usable probabilities: {probabilities!r}")
    return {name: value / total for name, value in scores.items()}


def _fallback_scores(question: str):
    """Keyword scoring, used only when Laya cannot be reached."""
    text = question.lower()
    web = sum(word in text for word in _WEB_HINTS)
    academic = sum(word in text for word in _ACADEMIC_HINTS)
    total = web + academic
    if total == 0:
        return {"tavily_search_node": 0.5, "arxiv_search_node": 0.5}
    web_score = web / total
    return {"tavily_search_node": web_score, "arxiv_search_node": 1 - web_score}


def research_decision_node(state: AgentState) -> Command:
    """
    Classifier node: Laya scores both retrieval tools and the higher score wins.

    Both calibrated probabilities and the margin are printed so the routing
    decision can be audited.
    """
    question = state.get("question", "")

    try:
        scores = _ask_laya(question)
        classifier = "Laya (calibrated probabilities)"
    except Exception as exc:  # Laya offline, timed out, or returned an error
        scores = _fallback_scores(question)
        classifier = "keyword fallback"
        print("[research_decision_node] WARNING: Laya unavailable, using keyword fallback")
        print("  error:", exc)

    web_score = scores["tavily_search_node"]
    academic_score = scores["arxiv_search_node"]
    route = "arxiv_search_node" if academic_score > web_score else "tavily_search_node"
    margin = abs(academic_score - web_score)

    print(f"[research_decision_node] classifier: {classifier}")
    print("  question:", question)
    print(f"  tavily_search_node   (web)      {web_score:6.3f}  {'#' * int(round(web_score * 40))}")
    print(f"  arxiv_search_node    (academic) {academic_score:6.3f}  {'#' * int(round(academic_score * 40))}")
    print(f"  -> chosen: {route}  (margin {margin:.3f})")

    return Command(
        goto=route,
        update={
            "research_route": route,
            "research_scores": scores,
            "research_reason": (
                f"{classifier}: {route} scored highest "
                f"(tavily={web_score:.3f}, arxiv={academic_score:.3f})"
            ),
        },
    )


## Define LangGraph Edges

In [12]:
from langgraph.graph import StateGraph, END, START

graph = StateGraph(AgentState)

# Nodes
graph.add_node("start", start_node)
graph.add_node("security", security_check_node)
graph.add_node("research_decision_node", research_decision_node)
graph.add_node("threat_detected", threat_detected_node)
graph.add_node("tavily_search_node", tavily_search_node)
graph.add_node("arxiv_search_node", arxiv_search_node)

# Start edge
graph.add_edge(START, "start")

# The custom input goes to the security layer next
graph.add_edge("start", "security")

# security_check_node returns a Command: a threat routes to "threat_detected",
# a clean input goes to "research_decision_node".
# research_decision_node returns a Command picking exactly one search node.
graph.add_edge("threat_detected", END)
graph.add_edge("tavily_search_node", END)
graph.add_edge("arxiv_search_node", END)

app = graph.compile()
